[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso-avanzados/01-revision-codigo.ipynb)

# Revisión Automática de Código con IA

> Aprende a construir un revisor de código automatizado con Claude que evalúa calidad,
> detecta problemas reales y genera tests unitarios automáticamente.

## Objetivos
- Implementar un revisor de código con rúbrica estructurada en JSON
- Detectar problemas de seguridad, legibilidad y eficiencia
- Generar sugerencias de mejora accionables
- Generar tests unitarios automáticamente para funciones Python

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic --quiet

## 2. Configuración inicial

In [ ]:
import anthropic
import json

# La API key se lee de la variable de entorno ANTHROPIC_API_KEY
client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print(f"Cliente inicializado. Modelo: {MODELO}")

## 3. Revisor de código con rúbrica JSON

El revisor evalúa cuatro dimensiones clave y devuelve un informe estructurado.

In [ ]:
RUBRICA_REVISION = """
Eres un revisor experto de código Python. Analiza el código proporcionado y devuelve
ÚNICAMENTE un objeto JSON con esta estructura exacta:
{
  "seguridad": <0-10>,
  "legibilidad": <0-10>,
  "eficiencia": <0-10>,
  "buenas_practicas": <0-10>,
  "puntuacion_total": <promedio>,
  "problemas": ["problema1", "problema2"],
  "sugerencias": ["sugerencia1", "sugerencia2"],
  "resumen": "<una línea con el veredicto general>"
}
"""

def revisar_codigo(codigo: str, contexto: str = "") -> dict:
    """Revisa código Python con Claude y devuelve informe estructurado."""
    prompt = f"{RUBRICA_REVISION}\n\nCódigo a revisar:\n```python\n{codigo}\n```"
    if contexto:
        prompt += f"\n\nContexto adicional: {contexto}"

    respuesta = client.messages.create(
        model=MODELO,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )

    texto = respuesta.content[0].text.strip()
    # Limpiar bloques de código markdown si los hay
    if "```" in texto:
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto)

print("Revisor de código listo.")

## 4. Ejemplos de código con diferente calidad

In [ ]:
# Código con múltiples problemas reales
CODIGO_PROBLEMATICO = '''
import subprocess

def buscar_usuario(nombre):
    # Busca usuario en la base de datos
    cmd = "SELECT * FROM usuarios WHERE nombre = '" + nombre + "'"
    resultado = subprocess.run(["mysql", "-e", cmd], capture_output=True)
    datos = []
    for x in resultado.stdout.decode().split("\\n"):
        if x != "":
            datos.append(x)
    return datos

def calcular(lista):
    s = 0
    for i in range(0, len(lista)):
        s = s + lista[i]
    return s / len(lista)
'''

# Código bien escrito
CODIGO_CORRECTO = '''
from typing import Optional
import sqlite3

def buscar_usuario(nombre: str, conn: sqlite3.Connection) -> list[dict]:
    """Busca usuarios por nombre usando consultas parametrizadas."""
    cursor = conn.cursor()
    cursor.execute("SELECT id, nombre, email FROM usuarios WHERE nombre = ?", (nombre,))
    columnas = [desc[0] for desc in cursor.description]
    return [dict(zip(columnas, fila)) for fila in cursor.fetchall()]

def calcular_media(valores: list[float]) -> Optional[float]:
    """Calcula la media aritmética de una lista de valores."""
    if not valores:
        return None
    return sum(valores) / len(valores)
'''

print("Revisando código problemático...")
informe_malo = revisar_codigo(CODIGO_PROBLEMATICO, "Aplicación web de gestión de usuarios")
print("\n=== CÓDIGO PROBLEMÁTICO ===")
print(f"Puntuación total: {informe_malo['puntuacion_total']}/10")
print(f"Problemas detectados: {len(informe_malo['problemas'])}")
for p in informe_malo['problemas']:
    print(f"  ⚠ {p}")

print("\nRevisando código correcto...")
informe_bueno = revisar_codigo(CODIGO_CORRECTO, "Aplicación web de gestión de usuarios")
print("\n=== CÓDIGO CORRECTO ===")
print(f"Puntuación total: {informe_bueno['puntuacion_total']}/10")
print(f"Resumen: {informe_bueno['resumen']}")

## 5. Comparativa detallada de métricas

In [ ]:
metricas = ["seguridad", "legibilidad", "eficiencia", "buenas_practicas"]

print(f"{'Métrica':<20} {'Problemático':>12} {'Correcto':>10}")
print("-" * 45)
for m in metricas:
    val_malo = informe_malo.get(m, 0)
    val_bueno = informe_bueno.get(m, 0)
    barra = "▓" * int(val_bueno) + "░" * (10 - int(val_bueno))
    print(f"{m:<20} {val_malo:>12.1f} {val_bueno:>10.1f}  {barra}")

print("-" * 45)
print(f"{'TOTAL':<20} {informe_malo['puntuacion_total']:>12.1f} {informe_bueno['puntuacion_total']:>10.1f}")

print("\nSugerencias para el código problemático:")
for i, s in enumerate(informe_malo['sugerencias'], 1):
    print(f"  {i}. {s}")

## 6. Generador automático de tests unitarios

In [ ]:
def generar_tests(codigo: str, nombre_funcion: str) -> str:
    """Genera tests unitarios con pytest para una función Python."""
    prompt = f"""Genera tests unitarios completos con pytest para la siguiente función Python.
Incluye casos normales, casos límite y casos de error.
Devuelve SOLO el código Python de los tests, sin explicaciones.

Función a testear:
```python
{codigo}
```

Genera tests para la función `{nombre_funcion}`."""

    respuesta = client.messages.create(
        model=MODELO,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    texto = respuesta.content[0].text.strip()
    # Extraer código de bloques markdown
    if "```python" in texto:
        texto = texto.split("```python")[1].split("```")[0].strip()
    elif "```" in texto:
        texto = texto.split("```")[1].split("```")[0].strip()
    return texto

FUNCION_A_TESTEAR = '''
def calcular_media(valores: list) -> float | None:
    """Calcula la media aritmética. Devuelve None si la lista está vacía."""
    if not valores:
        return None
    return sum(valores) / len(valores)
'''

tests = generar_tests(FUNCION_A_TESTEAR, "calcular_media")
print("=== TESTS GENERADOS AUTOMÁTICAMENTE ===")
print(tests)